In [1]:
from pathlib import Path
import csv
import json
import shutil
from collections import Counter

LAST_NAME = "Rojas"
STUDENT_ID = "TUPM-26-1244"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

TARGET_YEAR = 2015 + SEED_DIGIT
LOSS_FIELD = "Financial Loss (in Million $)"
COUNT_FIELD = "Attack Type"

print("Seed:", SEED_DIGIT)
print("Target year:", TARGET_YEAR)

Seed: 4
Target year: 2019


In [2]:
WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
print("Dataset found:", dataset_path.exists())

Dataset found: True


In [3]:
total_records = 0
invalid_losses = 0
fieldnames = []
selected = []        # (original_order, row, loss or None)
attack_counts = Counter()

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    fieldnames = reader.fieldnames
    for order, row in enumerate(reader):
        total_records += 1

        try:
            year = int((row.get("Year") or "").strip())
        except ValueError:
            continue

        if year != TARGET_YEAR:
            continue

        attack_counts[(row.get(COUNT_FIELD) or "Unknown").strip()] += 1

        try:
            loss = float(row[LOSS_FIELD])
        except (KeyError, ValueError, TypeError):
            loss = None
            invalid_losses += 1

        selected.append((order, row, loss))

valid = [item for item in selected if item[2] is not None]

total_loss = sum(item[2] for item in valid)
average_loss = total_loss / len(valid) if valid else 0

# sorted() is stable, so tied losses keep their original dataset order
top_five = sorted(valid, key=lambda item: item[2], reverse=True)[:5]

print("Target year:", TARGET_YEAR)
print("Total records in dataset:", total_records)
print("Incidents in target year:", len(selected))
print("Invalid financial values:", invalid_losses)
print("Count per attack type:", dict(attack_counts))
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(average_loss, 2))
print("Five costliest incidents:")
for rank, (order, row, loss) in enumerate(top_five, 1):
    print(f"  {rank}. {row.get(COUNT_FIELD)} | {row.get('Country')} | ${loss} M")
if not selected:
    print("No incidents found for the target year.")

Target year: 2019
Total records in dataset: 3000
Incidents in target year: 263
Invalid financial values: 0
Count per attack type: {'Phishing': 45, 'Ransomware': 38, 'Malware': 43, 'Man-in-the-Middle': 44, 'SQL Injection': 45, 'DDoS': 48}
Total financial loss (USD millions): 13134.69
Average financial loss (USD millions): 49.94
Five costliest incidents:
  1. Phishing | Germany | $99.97 M
  2. Man-in-the-Middle | Russia | $99.53 M
  3. Phishing | UK | $99.45 M
  4. Ransomware | Germany | $99.3 M
  5. SQL Injection | Australia | $99.2 M


In [4]:
filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_{TARGET_YEAR}.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_top5_incidents.json"
backup_path = BACKUP_DIR / json_path.name

with open(filtered_csv, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    for _, row, _ in selected:
        writer.writerow(row)

top_five_records = [
    {"rank": rank, "financial_loss_million_usd": loss, "record": row}
    for rank, (_, row, loss) in enumerate(top_five, 1)
]

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(top_five_records, file, indent=4)

shutil.copy2(json_path, backup_path)
backup_ok = backup_path.exists() and backup_path.stat().st_size == json_path.stat().st_size

print("Filtered CSV:", filtered_csv.name)
print("JSON summary file:", json_path.name)
print("Backup verification:", "VERIFIED" if backup_ok else "FAILED")

Filtered CSV: Rojas_TUPM-26-1244_incidents_2019.csv
JSON summary file: Rojas_TUPM-26-1244_top5_incidents.json
Backup verification: VERIFIED


In [5]:
report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incident_report.txt"

with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-----------------------------\n")
    report.write(f"Student: {LAST_NAME} ({STUDENT_ID})\n")
    report.write(f"Target Year: {TARGET_YEAR}\n")
    report.write(f"Incidents in Target Year: {len(selected)}\n\n")

    report.write("Incident Count per Attack Type:\n")
    for attack, count in attack_counts.most_common():
        report.write(f"  {attack}: {count}\n")

    report.write(f"\nTotal Financial Loss (USD millions): {total_loss:.2f}\n")
    report.write(f"Average Financial Loss (USD millions): {average_loss:.2f}\n\n")

    report.write("Five Costliest Incidents:\n")
    for rank, (_, row, loss) in enumerate(top_five, 1):
        report.write(f"  {rank}. {row.get(COUNT_FIELD)} - {row.get('Country')} - ${loss} M\n")

    report.write("\nOutput Files:\n")
    report.write(f"  Filtered CSV: {filtered_csv.name}\n")
    report.write(f"  JSON Summary: {json_path.name}\n")
    report.write(f"  Backup: {backup_path.name} (exists: {backup_path.exists()})\n")

print("Report created:", report_path)

Report created: output\Rojas_TUPM-26-1244_incident_report.txt


In [6]:
print("=== FINAL OUTPUT: CYBERSECURITY INCIDENT REPORT ===")
print("Target year:", TARGET_YEAR)
print("Total records in dataset:", total_records)
print("Incidents in target year:", len(selected))
print("Count per attack type:", dict(attack_counts))
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(average_loss, 2))
print("Filtered CSV file:", filtered_csv.name)
print("JSON summary file:", json_path.name)
print("Backup verification:", "VERIFIED" if backup_ok else "FAILED")
print("Report file:", report_path.name)

=== FINAL OUTPUT: CYBERSECURITY INCIDENT REPORT ===
Target year: 2019
Total records in dataset: 3000
Incidents in target year: 263
Count per attack type: {'Phishing': 45, 'Ransomware': 38, 'Malware': 43, 'Man-in-the-Middle': 44, 'SQL Injection': 45, 'DDoS': 48}
Total financial loss (USD millions): 13134.69
Average financial loss (USD millions): 49.94
Filtered CSV file: Rojas_TUPM-26-1244_incidents_2019.csv
JSON summary file: Rojas_TUPM-26-1244_top5_incidents.json
Backup verification: VERIFIED
Report file: Rojas_TUPM-26-1244_incident_report.txt
